In [5]:
import pandas as pd
import numpy as np

# 1. Load data mentah
df = pd.read_csv('../data//raw/train.csv')
df['date'] = pd.to_datetime(df['date'])

# sort berdasarkan store, item, lalu tanggal agar fungsi shift()
df = df.sort_values(['store', 'item', 'date']).reset_index(drop=True)

# 2. Fitur kalender (Time-based features)
df['day_of_week'] = df['date'].dt.dayofweek 
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['day_of_month'] = df['date'].dt.day
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year

# 3. Fitur lag (Lag features)
# Menggunakan groupby (['store', 'item']) agar data item 1 tidak bocor ke baris pertama Item 2
df['sales_lag_1'] = df.groupby(['store', 'item'])['sales'].shift(1) # Penjualan kemarin
df['sales_lag_7'] = df.groupby(['store', 'item'])['sales'].shift(7) # Penjualan hari yang sama minggu lalu
df['sales_lag_28'] = df.groupby(['store', 'item'])['sales'].shift(28) # Penjualan 4 minggu lalu

# 4. Fitur rolling (Rolling window)
# dengan catatan shift(1) dilakukan dulu sebelum rolling agar tidak bocor ke baris saat ini
df['rolling_mean_7'] = df.groupby(['store', 'item'])['sales'].transform(
    lambda x: x.shift(1).rolling(window=7).mean()
)
df['rolling_std_7'] = df.groupby(['store', 'item'])['sales'].transform(
    lambda x: x.shift(1).rolling(window=7).std()
)

# 5. Cek missing values dan hapus baris awawl akibat shift(28)
print("Jumlah NaN sebelum dibersihkan:\n", df.isna().sum())
df_clean = df.dropna().reset_index(drop=True)

# 6. Simpan ke folder processed
df_clean.to_parquet('../data/processed/train_features.parquet', index=False)
print("\nData berhasil disimpan! Ukuran akhir: ", df_clean.shape)
display(df_clean.head())

C:\Users\Rendy\AppData\Local\Temp\ipykernel_25552\2248580344.py:12: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df['day_of_week'] = df['date'].dt.dayofweek


Jumlah NaN sebelum dibersihkan:
 date                  0
store                 0
item                  0
sales                 0
day_of_week           0
is_weekend            0
day_of_month          0
month                 0
year                  0
sales_lag_1         500
sales_lag_7        3500
sales_lag_28      14000
rolling_mean_7     3500
rolling_std_7      3500
dtype: int64

Data berhasil disimpan! Ukuran akhir:  (899000, 14)


,date,store,item,sales,day_of_week,is_weekend,day_of_month,month,year,sales_lag_1,sales_lag_7,sales_lag_28,rolling_mean_7,rolling_std_7
0,2013-01-29,1,1,6,1,0,29,1,2013,11.0,7.0,13.0,10.428571,2.507133
1,2013-01-30,1,1,9,2,0,30,1,2013,6.0,9.0,11.0,10.285714,2.751623
2,2013-01-31,1,1,13,3,0,31,1,2013,9.0,8.0,14.0,10.285714,2.751623
3,2013-02-01,1,1,11,4,0,1,2,2013,13.0,14.0,13.0,11.000000,2.708013
4,2013-02-02,1,1,21,5,1,2,2,2013,11.0,12.0,10.0,10.571429,2.370453
